# GraphConPain — Interactive Demo

This notebook demonstrates inference, attention visualization, and feature importance analysis.

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
import numpy as np
import matplotlib.pyplot as plt

from models import GraphConPain
from evaluation.explainability import AttentionWeightAnalyzer, SHAPExplainer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT   = '../checkpoints/finetuned_full.pth'

model = GraphConPain()
if __import__('pathlib').Path(CKPT).exists():
    ckpt = torch.load(CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt.get('model_state_dict', ckpt), strict=False)
model = model.to(DEVICE).eval()
print('Model loaded.')

## Synthetic Episode Inference

In [ ]:
T = 30  # 1 second of data at 30 fps

batch = {
    'facial': torch.randn(1, T, 17).to(DEVICE),
    'body':   torch.randn(1, T, 51).to(DEVICE),
    'audio':  torch.randn(1, T, 65).to(DEVICE),
    'physio': torch.randn(1, T, 3, 250).to(DEVICE),
}

with torch.no_grad():
    preds = model(**batch)

PAIN_LEVELS = {0: 'None (0-2)', 1: 'Mild (3-4)', 2: 'Moderate (5-7)', 3: 'Severe (8-10)'}
probs = preds['class_logits'].exp().squeeze().cpu().numpy()
level = probs.argmax()

print(f'Continuous score:    {preds["continuous"].item():.2f}')
print(f'Pain level:          {PAIN_LEVELS[level]}')
print(f'Silent pain prob:    {preds["silent_logit"].sigmoid().item():.4f}')
print(f'Class probabilities: { {PAIN_LEVELS[i]: f"{p:.3f}" for i, p in enumerate(probs)} }')

## Attention Weight Visualization

In [ ]:
analyzer = AttentionWeightAnalyzer(model, DEVICE)
attn     = analyzer.get_attention_matrix(batch)
analyzer.plot_attention_heatmap(attn, title='Attention Matrix (Synthetic Episode)')
plt.show()

## Silent vs Vocal Pain Attention Patterns

In [ ]:
# Simulate vocal pain: high audio activity
batch_vocal = {
    'facial': torch.randn(1, T, 17).to(DEVICE),
    'body':   torch.randn(1, T, 51).to(DEVICE),
    'audio':  (torch.randn(1, T, 65) * 3).to(DEVICE),   # amplified audio
    'physio': torch.randn(1, T, 3, 250).to(DEVICE),
}

# Simulate silent pain: zero audio
batch_silent = {
    'facial': torch.randn(1, T, 17).to(DEVICE),
    'body':   (torch.randn(1, T, 51) * 2).to(DEVICE),   # amplified body
    'audio':  torch.zeros(1, T, 65).to(DEVICE),          # no audio
    'physio': torch.randn(1, T, 3, 250).to(DEVICE),
}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
MODS = ['Face', 'Body', 'Audio', 'Physio']

for ax, batch_ep, title in zip(axes, [batch_vocal, batch_silent],
                                ['Vocal Pain', 'Silent Pain']):
    attn = analyzer.get_attention_matrix(batch_ep)
    im   = ax.imshow(attn, cmap='YlOrRd', vmin=0, vmax=0.5)
    ax.set_xticks(range(4)); ax.set_xticklabels(MODS, fontsize=9)
    ax.set_yticks(range(4)); ax.set_yticklabels(MODS, fontsize=9)
    ax.set_title(title, fontweight='bold')
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{attn[i,j]:.2f}', ha='center', va='center', fontsize=8)
    plt.colorbar(im, ax=ax)

plt.suptitle('GAT Attention: Vocal vs Silent Pain', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()